<a href="https://colab.research.google.com/github/Ming-Silpakorn030/lab-ai-69/blob/main/lab_ai_week4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix


In [2]:
bc = load_breast_cancer()

In [3]:
X = bc.data
y = 1 - bc.target

In [5]:
X.shape

(569, 30)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

In [7]:
tree = DecisionTreeClassifier(random_state=42)

In [8]:
tree.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

In [10]:
pred = tree.predict(X_test)

In [11]:
print(confusion_matrix(y_test, pred))

[[100   7]
 [ 10  54]]


In [12]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import recall_score

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

print("accuracy เดามั่ว", dummy.score(X_test, y_test))
print("recall เดามั่ว", recall_score(y_test, dummy.predict(X_test)))

accuracy เดามั่ว 0.6257309941520468
recall เดามั่ว 0.0


In [13]:
from sklearn.metrics import classification_report
print(classification_report(y_test, pred,
      target_names=["ไม่เป็นมะเร็ง", "เป็นมะเร็ง"], digits=3))

               precision    recall  f1-score   support

ไม่เป็นมะเร็ง      0.909     0.935     0.922       107
   เป็นมะเร็ง      0.885     0.844     0.864        64

     accuracy                          0.901       171
    macro avg      0.897     0.889     0.893       171
 weighted avg      0.900     0.901     0.900       171



In [14]:
from sklearn.datasets import load_wine
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [15]:
wine = load_wine()
Xw, yw = wine.data, wine.target
Xw_train, Xw_test, yw_train, yw_test = train_test_split(
    Xw, yw, test_size=0.3, random_state=42)

In [16]:
knn_raw = KNeighborsClassifier()
knn_raw.fit(Xw_train, yw_train)
print("ไม่ปรับสเกล", knn_raw.score(Xw_test, yw_test))

ไม่ปรับสเกล 0.7407407407407407


In [17]:
knn_scaled = make_pipeline(StandardScaler(), KNeighborsClassifier())
knn_scaled.fit(Xw_train, yw_train)
print("ปรับสเกล", knn_scaled.score(Xw_test, yw_test))

ปรับสเกล 0.9629629629629629


In [18]:
from sklearn.model_selection import cross_val_score

pipe = make_pipeline(StandardScaler(), KNeighborsClassifier())
scores = cross_val_score(pipe, Xw, yw, cv=5)
print("คะแนน 5 รอบ", scores.round(4))
print("เฉลี่ย", scores.mean().round(4))

คะแนน 5 รอบ [0.9444 0.9444 0.9722 1.     0.8857]
เฉลี่ย 0.9494


In [19]:
import pandas as pd

rng = np.random.default_rng(42)
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df["region"] = rng.choice(["ไร่เหนือ", "ไร่กลาง", "ไร่ใต้"], size=len(df))
df["target"] = wine.target

In [20]:
for col in ["alcohol", "magnesium", "flavanoids", "proline"]:
    idx = rng.choice(len(df), size=15, replace=False)
    df.loc[idx, col] = np.nan

In [22]:
dups = df.sample(10, random_state=42)
df_dirty = pd.concat([df, dups], ignore_index=True)
df_dirty = df_dirty.sample(frac=1, random_state=42).reset_index(drop=True)
df_dirty.to_csv("wine_dirty.csv", index=False)

print("ขนาด", df_dirty.shape)
print("ค่าหายรวม", df_dirty.isna().sum().sum())
print("แถวซ้ำ", df_dirty.duplicated().sum())

ขนาด (188, 15)
ค่าหายรวม 62
แถวซ้ำ 10


In [23]:
df_dirty = pd.read_csv("wine_dirty.csv")
X_bad = df_dirty.drop(columns=["region", "target"]).values
y_bad = df_dirty["target"].values

In [25]:
pipe = make_pipeline(StandardScaler(), KNeighborsClassifier())

In [26]:
df_naive = df_dirty.dropna()
print("เหลือแถว", len(df_naive), "จาก", len(df_dirty))

เหลือแถว 137 จาก 188


In [27]:
df_clean = df_dirty.drop_duplicates().reset_index(drop=True)
print("หลังลบแถวซ้ำ", len(df_clean))

หลังลบแถวซ้ำ 178


In [28]:
df_clean = pd.get_dummies(df_clean, columns=["region"])

In [29]:
Xc = df_clean.drop(columns=["target"])
yc = df_clean["target"]
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, yc, test_size=0.3, random_state=42)

In [30]:
med = Xc_train.median(numeric_only=True)
Xc_train = Xc_train.fillna(med)
Xc_test = Xc_test.fillna(med)

In [31]:
pipe = make_pipeline(StandardScaler(), KNeighborsClassifier())
pipe.fit(Xc_train, yc_train)
print("คะแนนกองสอบ", round(pipe.score(Xc_test, yc_test), 4))

คะแนนกองสอบ 0.9259
